# 📖 Notebook 2: Cross-Region Replication

**Goal**: Understand how data is replicated between Azure paired regions for disaster recovery, and how failover works — all while keeping data within the EU.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why replication between paired regions is necessary
- The difference between synchronous and asynchronous replication
- How Azure handles failover between paired regions
- How to implement async replication that respects data residency

## 🛠️ Setup

Start the infrastructure first:

```bash
cd enterprise-patterns/gdpr-paired-regions
docker-compose up -d
```

### Visualization

- **Adminer**: http://localhost:8081  
  Open two tabs — one for EU-West (`postgres-eu-west:5432`, DB `gdpr_eu_west`) and one for EU-North (`postgres-eu-north:5432`, DB `gdpr_eu_north`)

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
import psycopg2
import time
from datetime import datetime

EU_WEST_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "database": "gdpr_eu_west",
    "user": "demo",
    "password": "demo"
}

EU_NORTH_CONFIG = {
    "host": "localhost",
    "port": 5434,
    "database": "gdpr_eu_north",
    "user": "demo",
    "password": "demo"
}

def get_connection(region):
    config = EU_WEST_CONFIG if region == "eu-west" else EU_NORTH_CONFIG
    return psycopg2.connect(**config)

# Test connections
for region in ["eu-west", "eu-north"]:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM users")
    print(f"✅ {region}: {cur.fetchone()[0]} users")
    conn.close()

✅ eu-west: 16 users
✅ eu-north: 16 users


## 1. Why Replicate Between Paired Regions?

Imagine your primary database is in **West Europe (Netherlands)**. What happens if:

- 🌊 The data center floods?
- ⚡ A power grid fails?
- 🔥 A fire destroys hardware?

Without replication, **all your data is gone**. With paired region replication:

```
Normal Operation:
  Users ──► EU-West (Primary) ──async──► EU-North (Replica)
                    │                          │
               Reads + Writes              Read-only backup

After Disaster in EU-West:
  Users ──────────────────────────────► EU-North (New Primary)
                                              │
                                     Now accepts writes
```

### Key Point for GDPR

Both EU-West and EU-North are in the **EU**. So even during failover, data **never leaves the EU**. This is why paired regions are always in the same geography.

## 2. Sync vs Async Replication

| Type | How It Works | Speed | Safety | Use Case |
|------|-------------|-------|--------|----------|
| **Synchronous** | Write waits for BOTH regions to confirm | Slow (100-200ms extra) | Zero data loss | Financial transactions |
| **Asynchronous** | Write confirms immediately, replica catches up later | Fast (no extra latency) | Small data loss possible | Most applications |

Azure uses **asynchronous replication** for paired regions because:
- Netherlands → Ireland is ~1000km, adding ~10ms network latency each way
- Sync replication would make every write 20ms+ slower
- For most apps, losing the last few seconds of data in a disaster is acceptable

Let's simulate both approaches:

In [2]:
# ── Simulating Asynchronous Replication ────────────────────
# In Azure, this is handled by the database service itself.
# Here we simulate it with application-level replication.

def replicate_user_async(user_id, source_region, target_region):
    """
    Simulates async replication from one region to another.
    
    In Azure, this is handled by:
    - Azure SQL Geo-Replication (for Azure SQL)
    - Azure Database for PostgreSQL Read Replicas
    - Azure Cosmos DB Multi-Region Writes
    
    We simulate it at the application level.
    """
    source_conn = get_connection(source_region)
    target_conn = get_connection(target_region)

    try:
        # Read from source
        src_cur = source_conn.cursor()
        src_cur.execute("""
            SELECT email, full_name, phone, date_of_birth, country_code,
                   home_region, consent_given, consent_date
            FROM users WHERE id = %s
        """, (user_id,))
        user_data = src_cur.fetchone()

        if not user_data:
            print(f"❌ User {user_id} not found in {source_region}")
            return

        # Simulate network latency (Netherlands → Ireland ≈ 10ms)
        print(f"   📡 Replicating across network ({source_region} → {target_region})...")
        time.sleep(0.01)  # 10ms simulated latency

        # Write to target (using UPSERT to handle re-runs)
        tgt_cur = target_conn.cursor()
        tgt_cur.execute("""
            INSERT INTO users (email, full_name, phone, date_of_birth, country_code,
                               home_region, consent_given, consent_date)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (email) DO UPDATE SET
                full_name = EXCLUDED.full_name,
                phone = EXCLUDED.phone,
                updated_at = NOW()
            RETURNING id
        """, user_data)
        replica_id = tgt_cur.fetchone()[0]

        # Log the replication event in BOTH regions
        for conn, region in [(source_conn, source_region), (target_conn, target_region)]:
            cur = conn.cursor()
            cur.execute("""
                INSERT INTO data_residency_log
                    (user_id, action, source_region, target_region, table_name, record_id, reason)
                VALUES (%s, 'replicate', %s, %s, 'users', %s,
                        'Async replication for disaster recovery — within EU geography')
            """, (user_id, source_region, target_region, user_id))
            conn.commit()

        print(f"   ✅ User '{user_data[1]}' replicated to {target_region} (replica ID: {replica_id})")
        return replica_id

    finally:
        source_conn.close()
        target_conn.close()


# Demo: Replicate EU-West users to EU-North
print("🔄 Async Replication: EU-West → EU-North")
print("=" * 50)
print("(Simulates Azure SQL Geo-Replication)\n")

# Get some EU-West users to replicate
conn = get_connection("eu-west")
cur = conn.cursor()
cur.execute("SELECT id, full_name FROM users WHERE home_region = 'eu-west' LIMIT 3")
users_to_replicate = cur.fetchall()
conn.close()

for user_id, name in users_to_replicate:
    print(f"\n👤 Replicating user {user_id} ({name}):")
    replicate_user_async(user_id, "eu-west", "eu-north")

🔄 Async Replication: EU-West → EU-North
(Simulates Azure SQL Geo-Replication)


👤 Replicating user 1 (Anna de Vries):
   📡 Replicating across network (eu-west → eu-north)...
   ✅ User 'Anna de Vries' replicated to eu-north (replica ID: 1)

👤 Replicating user 2 (Marc Dupont):


   📡 Replicating across network (eu-west → eu-north)...


   ✅ User 'Marc Dupont' replicated to eu-north (replica ID: 2)

👤 Replicating user 3 (Claire Martin):
   📡 Replicating across network (eu-west → eu-north)...


   ✅ User 'Claire Martin' replicated to eu-north (replica ID: 3)


## 3. Replication Lag — The Tradeoff

With async replication, there's always a **replication lag** — a window of time where the replica is behind the primary.

```
Timeline:
  t=0ms    User writes to EU-West       ✅ Primary has data
  t=0ms    Write confirmed to user       ⏳ Replica doesn't have it yet
  t=10ms   Data arrives at EU-North      ✅ Replica has data
  
  If EU-West fails between t=0ms and t=10ms, those 10ms of data are LOST.
```

Azure's typical replication lag for paired regions is **< 5 seconds**.

Let's measure the lag in our simulation:

In [3]:
# ── Measuring Replication Lag ──────────────────────────────

def measure_replication_lag():
    """
    Writes a record to EU-West and measures how long until
    it appears in EU-North.
    """
    # Write to EU-West
    west_conn = get_connection("eu-west")
    west_cur = west_conn.cursor()

    timestamp = datetime.now().isoformat()
    test_email = f"lag-test-{timestamp}@example.com"

    write_start = time.time()
    west_cur.execute("""
        INSERT INTO users (email, full_name, phone, country_code, home_region, consent_given)
        VALUES (%s, 'Lag Test User', '+1-555-0000', 'NL', 'eu-west', TRUE)
        RETURNING id
    """, (test_email,))
    user_id = west_cur.fetchone()[0]
    west_conn.commit()
    write_time = (time.time() - write_start) * 1000

    print(f"✏️  Write to EU-West: {write_time:.1f}ms (user_id={user_id})")

    # Now replicate (this is what Azure does automatically)
    repl_start = time.time()
    replicate_user_async(user_id, "eu-west", "eu-north")
    repl_time = (time.time() - repl_start) * 1000

    print(f"⏱️  Replication lag: {repl_time:.1f}ms")
    print(f"   Total time until data exists in both regions: {write_time + repl_time:.1f}ms")

    # Clean up test user
    for region in ["eu-west", "eu-north"]:
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("DELETE FROM users WHERE email = %s", (test_email,))
        conn.commit()
        conn.close()

    west_conn.close()

print("📊 Replication Lag Test")
print("=" * 50)
measure_replication_lag()

📊 Replication Lag Test
✏️  Write to EU-West: 3.0ms (user_id=18)


   📡 Replicating across network (eu-west → eu-north)...
   ✅ User 'Lag Test User' replicated to eu-north (replica ID: 20)
⏱️  Replication lag: 131.4ms
   Total time until data exists in both regions: 134.4ms


## 4. Failover Simulation

When the primary region goes down, the system must **failover** to the paired region. Here's how Azure handles it:

1. Azure detects the primary is unhealthy
2. DNS is updated to point to the secondary region
3. The secondary is promoted to primary (accepts writes)
4. When the original region recovers, it becomes the new secondary

Let's simulate this:

In [4]:
# ── Failover Simulation ────────────────────────────────────

class PairedRegionManager:
    """
    Simulates Azure's paired region failover mechanism.
    
    In Azure, this is handled by:
    - Azure Traffic Manager (DNS-based failover)
    - Azure SQL Auto-Failover Groups
    - Azure Front Door (HTTP-based routing)
    """

    def __init__(self):
        self.primary = "eu-west"
        self.secondary = "eu-north"
        self.is_primary_healthy = True

    def get_active_region(self):
        """Returns the region that should handle writes."""
        if self.is_primary_healthy:
            return self.primary
        return self.secondary

    def health_check(self, region):
        """Checks if a region's database is reachable."""
        try:
            conn = get_connection(region)
            cur = conn.cursor()
            cur.execute("SELECT 1")
            conn.close()
            return True
        except Exception:
            return False

    def simulate_failure(self):
        """Simulates EU-West going down."""
        print("\n🔥 DISASTER: EU-West region is DOWN!")
        print("   (In reality: data center flood, power outage, etc.)")
        self.is_primary_healthy = False
        print(f"   Active region switched: {self.primary} → {self.secondary}")
        print(f"   All writes now go to {self.secondary}")

    def simulate_recovery(self):
        """Simulates EU-West coming back online."""
        print("\n✅ RECOVERY: EU-West region is back online!")
        self.is_primary_healthy = True
        print(f"   Active region switched back: {self.secondary} → {self.primary}")
        print(f"   Writes return to {self.primary}")

    def write_user(self, email, full_name, country_code):
        """Writes a user to whichever region is currently active."""
        active = self.get_active_region()
        conn = get_connection(active)
        cur = conn.cursor()

        cur.execute("""
            INSERT INTO users (email, full_name, country_code, home_region, consent_given, consent_date)
            VALUES (%s, %s, %s, %s, TRUE, NOW())
            ON CONFLICT (email) DO NOTHING
            RETURNING id
        """, (email, full_name, country_code, active))

        result = cur.fetchone()
        conn.commit()
        conn.close()

        if result:
            print(f"   ✏️  Wrote '{full_name}' to {active} (ID: {result[0]})")
        else:
            print(f"   ⏭️  '{full_name}' already exists in {active}")
        return active


# ── Run the failover scenario ──────────────────────────────

manager = PairedRegionManager()

print("🏗️  Paired Region Failover Simulation")
print("=" * 50)

# Phase 1: Normal operation
print("\n📍 Phase 1: Normal Operation")
print(f"   Active region: {manager.get_active_region()}")
manager.write_user("failover.test1@example.de", "Test User 1", "DE")

# Phase 2: EU-West goes down
manager.simulate_failure()
print(f"\n📍 Phase 2: Failover Active")
print(f"   Active region: {manager.get_active_region()}")
manager.write_user("failover.test2@example.de", "Test User 2", "DE")

# Phase 3: EU-West recovers
manager.simulate_recovery()
print(f"\n📍 Phase 3: Recovery Complete")
print(f"   Active region: {manager.get_active_region()}")
manager.write_user("failover.test3@example.de", "Test User 3", "DE")

print("\n💡 Key insight: During ALL phases, data stayed within the EU!")
print("   EU-West (Netherlands) and EU-North (Ireland) are both in the EU.")

🏗️  Paired Region Failover Simulation

📍 Phase 1: Normal Operation
   Active region: eu-west
   ✏️  Wrote 'Test User 1' to eu-west (ID: 19)

🔥 DISASTER: EU-West region is DOWN!
   (In reality: data center flood, power outage, etc.)
   Active region switched: eu-west → eu-north
   All writes now go to eu-north

📍 Phase 2: Failover Active
   Active region: eu-north
   ✏️  Wrote 'Test User 2' to eu-north (ID: 21)

✅ RECOVERY: EU-West region is back online!
   Active region switched back: eu-north → eu-west
   Writes return to eu-west

📍 Phase 3: Recovery Complete
   Active region: eu-west


   ✏️  Wrote 'Test User 3' to eu-west (ID: 20)

💡 Key insight: During ALL phases, data stayed within the EU!
   EU-West (Netherlands) and EU-North (Ireland) are both in the EU.


In [5]:
# ── Verify: Where did each write land? ─────────────────────

print("🔍 Verifying Write Locations After Failover")
print("=" * 60)

for region in ["eu-west", "eu-north"]:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("""
        SELECT email, full_name, home_region
        FROM users
        WHERE email LIKE 'failover.test%'
        ORDER BY email
    """)
    results = cur.fetchall()
    print(f"\n📦 {region.upper()}:")
    for row in results:
        print(f"   {row[1]} ({row[0]}) — stored in {row[2]}")
    if not results:
        print("   (no failover test users here)")
    conn.close()

print("\n💡 Notice:")
print("   - Test User 1 & 3 → EU-West (primary was healthy)")
print("   - Test User 2 → EU-North (written during failover)")
print("   - After recovery, Test User 2 should be replicated back to EU-West")

🔍 Verifying Write Locations After Failover

📦 EU-WEST:
   Test User 1 (failover.test1@example.de) — stored in eu-west
   Test User 3 (failover.test3@example.de) — stored in eu-west

📦 EU-NORTH:
   Test User 2 (failover.test2@example.de) — stored in eu-north

💡 Notice:
   - Test User 1 & 3 → EU-West (primary was healthy)
   - Test User 2 → EU-North (written during failover)
   - After recovery, Test User 2 should be replicated back to EU-West


## 5. Why Microsoft Uses Paired Region Replication

### Azure's Guarantees

| Feature | Guarantee |
|---------|----------|
| **Recovery Point Objective (RPO)** | < 5 seconds of data loss |
| **Recovery Time Objective (RTO)** | < 30 minutes to failover |
| **Data residency** | Never leaves the geography |
| **Update rollouts** | Paired regions updated sequentially (never both at once) |

### The Sequential Update Trick

Azure never updates both paired regions at the same time. If an update breaks West Europe, North Europe is still on the old (working) version. This gives Microsoft time to fix the issue before it affects the pair.

### Cost Implications

Geo-replication costs money (you're running two databases), but for EU enterprise customers, it's non-negotiable:
- GDPR fines can be up to **4% of global revenue**
- Downtime costs enterprise customers **€5,000-€100,000 per hour**
- The cost of two databases is trivial compared to these risks

In [6]:
# ── Clean up failover test data ────────────────────────────

for region in ["eu-west", "eu-north"]:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("DELETE FROM users WHERE email LIKE 'failover.test%'")
    deleted = cur.rowcount
    conn.commit()
    conn.close()
    if deleted > 0:
        print(f"🧹 Cleaned up {deleted} test users from {region}")

print("✅ Test data cleaned up")

🧹 Cleaned up 2 test users from eu-west


🧹 Cleaned up 1 test users from eu-north
✅ Test data cleaned up


## 🎯 Key Takeaways

1. **Paired regions replicate data within the same geography** — EU data stays in the EU
2. **Async replication** is the default — small lag is acceptable for most workloads
3. **Failover** automatically switches writes to the secondary if the primary goes down
4. **Both regions are always within GDPR jurisdiction** — no compliance violation during DR
5. **Replication lag** is the window of potential data loss — typically < 5 seconds in Azure

## ⏭️ Next Up

In **Notebook 3**, we'll tackle the most challenging GDPR requirement: the **Right to Erasure** (Article 17) — implementing "right to be forgotten" with cascading deletes across both regions.